# **FRAP Batch Analysis Pipeline**
---



# **Prepare environment and load dependencies**

In [ ]:
#Install python packages
!pip install czitools
!pip install pylibCZIrw

!pip install bioio
!pip install bioio-czi
!pip install scikit_posthocs
!pip install zarr
!pip install statannotations

#Import libraries
import sys
sys.path.append('tools')
import frap_tools
import io_tools as io_tools
import processing_tools as processing_tools
from matplotlib import pyplot as plt
import pandas as pd
import os
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter


# Run analysis

## **Define input folder**

In [ ]:

input_folder_paths = ["../sample_images/CSV/MUT",
                      "../sample_images/CSV/MUT2",
                      "../sample_images/CSV/WT"]



output_path = "../sample_images/CSV/test_output"

do_whole_cell = False  #False = reference from czi file ROI / True =  use whole cell as reference
exp_fitting_order = 1 # 1 for monoexponential /  2 for biexponential




#Check data integrity
for idx, f in enumerate(input_folder_paths):
  if os.path.isdir(f):
      print(f"Input folder :'{f}'")
  else:
      print('\033[31m' + f"WARNING: The folder '{f}' does not exist." + '\033[0m')

if os.path.isdir(output_path):
      print(f"Path to output data: '{f}'")
else:
      print('\033[31m' + f"WARNING: The output folder '{f}' does not exist." + '\033[0m')

## Launch analysis and plot results

In [ ]:

roiData = []
frap_experiment = []
wt_preview = []

for idx,folderPath in enumerate(input_folder_paths):

    #print('Processing files from directory: ' + folderPath)
    dataset_roiData, dataset_frap_experiment, wt_preview = frap_tools.process_FRAP_folder(folderPath, wcell_corr= do_whole_cell, fitting_exp = exp_fitting_order)
    print(os.path.dirname(folderPath))
    dataset_roiData.insert(loc=0, column = 'folder', value = os.path.basename(folderPath))
    dataset_frap_experiment.insert(loc=0, column = 'folder', value = os.path.basename(folderPath))
    roiData.append(dataset_roiData)
    frap_experiment.append(dataset_frap_experiment)
    io_tools.saveResults(os.path.join(output_path,os.path.basename(folderPath)), dataset_roiData, dataset_frap_experiment)



nrows = 6 + 2* len(input_folder_paths)
nd = len(input_folder_paths)
ax = []
#Color palette for multiple groups
palette = sns.color_palette("tab10")
#color palette for dishes
n_base_colors = 15
base_palette = sns.color_palette("husl", n_base_colors)
dish_palette = {str(i + 1): base_palette[i % n_base_colors] for i in range(200)}



#PLOT results
fig = plt.figure(constrained_layout=True, figsize=(12, nrows*3))
# hspace is passed directly to the gridspec (rather than via plt.subplots_adjust)
# since the constrained_layout engine ignores/warns on subplots_adjust
spec = fig.add_gridspec(nrows, 4, hspace=0.4)

# PLOT raw data per folder and dish
for i in range(nd):
    ax = fig.add_subplot(spec[i, :])
    sns.lineplot(ax = ax, data =roiData[i], x= 'timestamp_frap_r', y = 'frap_norm_r', hue = 'dish', palette=dish_palette )
    ax.set_title(os.path.basename(input_folder_paths[i]) + ' per dish recovery curves', parse_math=False)
    ax.set_ylabel('Raw intensity')
    ax.set_xlabel('Time[s]')
    ax.margins(x=0)
    sns.move_legend(ax, "lower right", frameon=False)
#ax.legend(title = 'dish',frameon=False)


# PLOT raw data all together
ax = fig.add_subplot(spec[nd, :])
for i in range(nd):
        sns.lineplot(ax = ax, data =roiData[i], x= 'timestamp_frap_r', y = 'frap_norm_r',  label = os.path.basename(input_folder_paths[i]) , color=palette[i])
ax.set_title(os.path.basename(input_folder_paths[i]) + ' per dish recovery curves', parse_math=False)
ax.set_ylabel('Raw intensity')
ax.set_xlabel('Time[s]')
ax.margins(x=0)
sns.move_legend(ax, "lower right", frameon=False)

# PLOT Boxplots for mobile fraction and tau1/2
ax = fig.add_subplot(spec[nd+1: nd+3,0:2])
for i in range(nd):
    sns.boxplot(ax = ax, data = frap_experiment[i], x="folder", y="mob", label = os.path.basename(input_folder_paths[i]), color=palette[i])
ax.set_title('Mobile fraction')
ax.set_ylabel('Mobile fraction')
ax.set_xlabel(None)
#ax.set(ylim=( 0,1.3))
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1),frameon=False)

ax = fig.add_subplot(spec[nd+1:nd+3,2::])
for i in range(nd):
    sns.boxplot(ax = ax, data = frap_experiment[i], x="folder", y="half_max", label = os.path.basename(input_folder_paths[i]), color=palette[i])
ax.set_title('Recovery half max ')
ax.set_ylabel(r'$\tau^{1/2}$ [s]')
ax.set_xlabel(None)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1),frameon=False)
#ax.set(ylim=( 0,4))



# PLOT Mobile fraction per dish
ax = fig.add_subplot(spec[nd+3, :])
for i in range(nd):
    #Fix bug with seaborn where if there is a single dish the dodge parameter needs to be set to false for pointplot or will crash
    n_dish=len(frap_experiment[i]['dish'].unique())
    if n_dish == 1: dodge_param = False
    else: dodge_param = .4

    sns.stripplot(ax = ax, data = frap_experiment[i], x="folder", y="mob", hue = "dish",
        dodge=dodge_param, alpha=.2, legend=False, palette = dish_palette)
    sns.pointplot(ax = ax, data = frap_experiment[i], x="folder", y="mob", hue = "dish",
    dodge=dodge_param, linestyle="none", errorbar=None, marker="_", markersize=20, markeredgewidth=3,  palette=dish_palette)
ax.set_title('Mobile fraction per dish')
ax.set_ylabel('Mobile fraction')
ax.set_xlabel(None)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1),frameon=False)
#plt.show()


# PLOT tau 1/2  per dish
ax = fig.add_subplot(spec[nd+4, :])
for i in range(nd):
    #Fix bug with seaborn where if there is a single dish the dodge parameter needs to be set to false for pointplot or will crash
    n_dish=len(frap_experiment[i]['dish'].unique())
    if n_dish == 1: dodge_param = False
    else: dodge_param = .4

    sns.stripplot(ax = ax, data = frap_experiment[i], x="folder", y="half_max", hue = "dish",
        dodge=dodge_param, alpha=.2, legend=False, palette="Set2" )

    sns.pointplot(ax = ax, data = frap_experiment[i], x="folder", y="half_max", hue = "dish",
    dodge=dodge_param, linestyle="none", errorbar=None, marker="_", markersize=20, markeredgewidth=3, palette="Set2")
ax.set_title('Recovery half max per dish')
ax.set_ylabel(r'$\tau^{1/2}$ [s]')
ax.set_xlabel(None)
ax.legend(title='dish')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1),frameon=False)


# PLOT FRAP recovery full scale per group
ax = fig.add_subplot(spec[nd+5 , :])
for i in range(nd):
    sns.lineplot(ax = ax, data =roiData[i], x= 'timestamp_frap_r', y = 'frap_fullscale_norm_r', label = os.path.basename(input_folder_paths[i]), errorbar = 'se')
    #sns.lineplot(ax = ax, data =roiData[i], x= 'timestamp_frap_r', y = 'frap_fullscale_norm_r', label = os.path.basename(input_folder_paths[i]) ,errorbar = 'se')
ax.set_title('Full scale FRAP recovery')
ax.set_ylabel('Full scale Normalized intensity')
ax.set_xlabel('Time[s]')
ax.margins(x=0)
sns.move_legend(ax, "lower right", frameon=False)


# PLOT FRAP recovery full scale per group and dish
for i in range(nd):
    ax = fig.add_subplot(spec[nd+6 + i, :])
    sns.scatterplot(ax = ax, data =roiData[i][roiData[i]['timepoint_frap']>=0], x= 'timestamp_frap_r', y = 'frap_norm_r',
                alpha = 0.1, color=palette[i], legend = None)
    sns.lineplot(ax = ax, data =roiData[i][roiData[i]['timepoint_frap']>=0], x= 'timestamp_frap_r', y = 'frap_norm_r', label = os.path.basename(input_folder_paths[i]),
                errorbar = 'sd', err_style = 'bars', color=palette[i])
    ax.set_title('Full scale recovery - ' + os.path.basename(input_folder_paths[i]), parse_math=False)
    ax.set_ylabel('Full scale Normalized intensity')
    ax.set_xlabel('Time[s]')
    sns.move_legend(ax, "lower right", frameon=False)


basenames =  [os.path.basename(f) for f in input_folder_paths]
outputName = "_".join(basenames)

fig.savefig(os.path.join(output_path,outputName + 'frap_results.pdf'), bbox_inches='tight', dpi=300)


In [ ]:
basenames = [os.path.basename(f) for f in input_folder_paths]

processing_tools.run_dish_aggregated_comparison(frap_experiment, basenames, 'mob', 'Mobile Fraction')
processing_tools.run_dish_aggregated_comparison(frap_experiment, basenames, 'half_max', 'Recovery Half Max')


##Build QC plots for each analyzed experiment

In [ ]:
# QC: per-file diagnostic plots (reference fit + recovery curve) as a single PDF

# Combine per-folder roiData / frap_experiment into single long DataFrames,
# one row per timepoint (roiData) / one row per file (frap_experiment)
roiData_all = pd.concat(roiData, ignore_index=True)
frap_experiment_all = pd.concat(frap_experiment, ignore_index=True)
files = sorted(roiData_all['file'].unique())
n_files = len(files)

fig, axes = plt.subplots(n_files, 2, figsize=(12, 3 * n_files), squeeze=False)

for i, f in enumerate(files):
    df_f = roiData_all[roiData_all['file'] == f]

    # Label with folder/dish too, if available, for easier traceability
    label_bits = [f]
    if 'folder' in df_f.columns:
        label_bits.append(str(df_f['folder'].iloc[0]))
    if 'dish' in df_f.columns:
        label_bits.append(f"dish {df_f['dish'].iloc[0]}")
    row_label = " | ".join(label_bits)

    # Matching row in frap_experiment for this file (mob / half_max / recovery fit values)
    fit_row = frap_experiment_all[frap_experiment_all['file'] == f]

    # Left column: reference (points) & reference_synth (line) vs full timestamp_frap range
    ax_left = axes[i, 0]
    ax_left.plot(df_f['timestamp_frap'], df_f['reference'], marker='o', linestyle='-',
                 markersize=3.0, linewidth=1, color='tab:gray', label='reference')
    ax_left.plot(df_f['timestamp_frap'], df_f['reference_synth'], color='tab:red',
                 label='reference_synth')
    # parse_math=False: row_label comes from filenames/folders, which may contain
    # characters (e.g. '#', an even count of '$') that matplotlib would otherwise
    # misinterpret as mathtext and fail to parse.
    ax_left.set_title(row_label + '\nreference', fontsize=9, parse_math=False)
    ax_left.set_xlabel('Time [s]')
    ax_left.set_ylabel('Intensity')
    ax_left.margins(x=0)
    ax_left.legend(frameon=False, fontsize=7, loc='best')

    # Right column: bleach_recovery_curve (line) & frap_norm (points), timestamp_frap > 0 only
    df_pos = df_f[df_f['timestamp_frap'] > 0]
    ax_right = axes[i, 1]
    ax_right.plot(df_pos['timestamp_frap'], df_pos['bleach_recovery_curve'],
                  color='tab:orange', label='bleach_recovery_curve')
    ax_right.plot(df_pos['timestamp_frap'], df_pos['frap_norm'], marker='o', linestyle='-',
                  markersize=3.0, linewidth=1, color='tab:blue', label='frap_norm')

    if not fit_row.empty:
        # Mark fitted mobile fraction (horizontal) and recovery half-max time (vertical)
        mob_val = fit_row['mob'].iloc[0]
        half_max_val = fit_row['half_max'].iloc[0]
        ax_right.axhline(mob_val, color='tab:green', linestyle='--', linewidth=1,
                          label=f'mob = {mob_val:.2f}')
        ax_right.axvline(half_max_val, color='tab:purple', linestyle='--', linewidth=1,
                          label=f'half_max = {half_max_val:.2f}')

        # Print the fitted exponential recovery equation and its R^2
        y0, A, tau = fit_row['recovery_fit'].iloc[0]
        r2 = fit_row['recovery_fit_r2'].iloc[0]
        eq_text = (rf"$Y = {y0:.3f} + {A:.3f}\,e^{{-{tau:.3f}\,t}}$" "\n"
                   f"$R^2 = {r2:.3f}$")
        ax_right.text(0.98, 0.02, eq_text, transform=ax_right.transAxes,
                      fontsize=7, ha='right', va='bottom',
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.7'))

    ax_right.set_title(row_label + '\nrecovery', fontsize=9, parse_math=False)
    ax_right.set_xlabel('Time [s]')
    ax_right.set_ylabel('Normalized intensity')
    ax_right.margins(x=0)
    ax_right.legend(frameon=False, fontsize=7, loc='best')

plt.tight_layout()

qc_pdf_path = os.path.join(output_path, 'QC_per_file.pdf')
fig.savefig(qc_pdf_path, bbox_inches='tight', dpi=150)
plt.close(fig)

print(f"QC PDF saved to: {qc_pdf_path}  ({n_files} files)")